# 02 — Train U-Net for TEM Grain Boundary Detection

**Goal:** Train a U-Net (ResNet-18 encoder) to detect grain boundaries in cropped TEM layer images. After inference, watershed splits boundaries into individual grains.

**Environment:** Google Colab (GPU). Runtime → Change runtime type → T4 GPU.

**Data needed:** Upload `training_data/grain_segmentation/` to Google Drive (or directly to Colab).

**How geometry is handled:**
- The exported layer crops can have any size — training cuts random **512×512 tiles** out of them (native resolution, no resize), so batches never break on mismatched shapes.
- Pixels **outside the layer** (neighboring layers, vacuum) are marked `255` in the boundary target and **excluded from the loss** (`ignore_index`). This requires `layer_masks/` in the dataset (exported by the app ZIP and by `export_training_data.py`). Without it the whole image is used as training signal (a warning is printed).
- Flips / `RandomRotate90` keep rotational diversity lossless — the image is never derotated.


## 1. Install dependencies

In [ ]:
!pip install segmentation-models-pytorch
!pip install tifffile imagecodecs albumentations

## 2. Load training data

Expected structure:
```
grain_segmentation/
├── images/       # cropped layer images (grayscale TIFF)
├── boundaries/   # boundary targets (0 = no boundary, 255 = boundary)
├── labels/       # labeled images (0 = bg, 1 = grain 1, ...) — eval only
├── layer_masks/  # 255 = inside the layer, 0 = outside — optional but recommended
└── train_val_split.json
```

`layer_masks/` is written by the app's "Export Training Data" grain ZIP (`grain_layer_mask.tif`) and by `export_training_data.py`. If it is missing, every pixel is treated as valid.


In [ ]:
import os, json, numpy as np, tifffile, cv2
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

DATA_DIR = Path('/content/grain_segmentation')
IMG_SIZE = 512          # training tile size (random crops at native resolution)
IGNORE_INDEX = 255      # target value for pixels outside the layer (excluded from loss)

with open(DATA_DIR / 'train_val_split.json') as f:
    split = json.load(f)
print(f'Train: {len(split["train"])}, Val: {len(split["val"])}')


## 3. Dataset + augmentation

In [ ]:
HAS_LAYER_MASKS = (DATA_DIR / 'layer_masks').exists()

# Training: pad (reflect) if the crop is smaller than the tile, then cut a random
# 512×512 tile. Random crop position differs every epoch → more effective samples.
train_tf = A.Compose([
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=cv2.BORDER_REFLECT_101),
    A.RandomCrop(height=IMG_SIZE, width=IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.Normalize(mean=0.5, std=0.5),
    ToTensorV2(),
])

# Validation/inference: keep the full image — U-Net is fully convolutional and
# accepts any size; just pad to a multiple of 32 (ResNet-18 has 5 downsampling
# steps), then crop the prediction back.
val_tf = A.Compose([
    A.PadIfNeeded(min_height=None, min_width=None,
                  pad_height_divisor=32, pad_width_divisor=32,
                  border_mode=cv2.BORDER_REFLECT_101),
    A.Normalize(mean=0.5, std=0.5),
    ToTensorV2(),
])


class GrainDataset(Dataset):
    """Variable-size exported crops -> fixed-size tensors.

    Pixels outside the layer (layer_masks/{id}.tif == 0) get target value
    IGNORE_INDEX (255) and do not contribute to the loss — this replaces the
    mean-fill of neighboring layers done by the old exporter. If no layer
    masks are present, every pixel is treated as valid.
    """

    def __init__(self, data_dir, sample_ids, augment=False):
        self.data_dir = Path(data_dir)
        self.sample_ids = list(sample_ids)
        self.transform = train_tf if augment else val_tf

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        img = tifffile.imread(self.data_dir / 'images' / f'{sample_id}.tif')
        if img.ndim == 3:
            img = img[..., 0]
        boundary = tifffile.imread(self.data_dir / 'boundaries' / f'{sample_id}.tif')
        boundary = (boundary > 127).astype(np.float32)

        lm_path = self.data_dir / 'layer_masks' / f'{sample_id}.tif'
        if lm_path.exists():
            valid = tifffile.imread(lm_path) > 127
            if (~valid).any():
                boundary[~valid] = IGNORE_INDEX

        img_3ch = np.stack([img, img, img], axis=-1)
        augmented = self.transform(image=img_3ch, mask=boundary)
        return augmented['image'], augmented['mask'].unsqueeze(0).float(), sample_id


train_ds = GrainDataset(DATA_DIR, split['train'], augment=True)
val_ds = GrainDataset(DATA_DIR, split['val'], augment=False)
# batch_size=8 fits a T4 at 512×512; reduce to 4 if you hit OOM.
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
if not HAS_LAYER_MASKS:
    print('WARNING: layer_masks/ not found — the whole image will be used as '
          'training signal. Re-export the data (app ZIP or export_training_data.py) '
          'to include layer masks.')


## 4. Model: U-Net + ResNet-18

In [ ]:
import segmentation_models_pytorch as smp

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = smp.Unet(
    encoder_name='resnet18',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    # NB: no activation here — the model returns raw logits; the losses below
    # apply sigmoid internally (from_logits=True). Applying sigmoid twice (model
    # + loss) silently degrades training.
).to(device)

dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True,
                                ignore_index=IGNORE_INDEX)
bce_loss = smp.losses.SoftBCEWithLogitsLoss(ignore_index=IGNORE_INDEX)
loss_fn = lambda pred, target: 0.5 * dice_loss(pred, target) + 0.5 * bce_loss(pred, target)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')


## 5. Training loop

In [ ]:
from tqdm import tqdm

N_EPOCHS = 50
best_val_loss = float('inf')
CHECKPOINT_PATH = '/content/unet_grain_boundaries.pth'

for epoch in range(N_EPOCHS):
    # Train — each step sees a random 512×512 tile of a random image.
    model.train()
    train_loss = 0
    for imgs, masks, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{N_EPOCHS} [train]'):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = loss_fn(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate — full images (batch size 1, sizes may differ);
    # pixels outside the layer are excluded by loss_fn via ignore_index.
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks, _ in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            val_loss += loss_fn(preds, masks).item()
    val_loss /= len(val_loader)

    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'  → Saved best model (val_loss={val_loss:.4f})')

print(f'\nTraining complete. Best val_loss: {best_val_loss:.4f}')


## 6. Evaluate: boundaries → watershed → grains

In [ ]:
from scipy import ndimage as ndi
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()


def normalize_img(img: np.ndarray) -> np.ndarray:
    """Same normalization as training (A.Normalize(mean=0.5, std=0.5))."""
    x = img.astype(np.float32)
    if x.max() > 255:  # safety: scale down non-8-bit input
        x = x / 257.0
    x = x / 255.0
    return (x - 0.5) / 0.5


def predict_boundaries(img: np.ndarray) -> np.ndarray:
    """Full-image inference: pad to /32 (reflect), forward, sigmoid, crop back."""
    h, w = img.shape[:2]
    ph = (32 - h % 32) % 32
    pw = (32 - w % 32) % 32
    img_3ch = np.stack([img, img, img], axis=-1)
    padded = np.pad(img_3ch, ((0, ph), (0, pw), (0, 0)), mode='reflect')
    x = torch.from_numpy(normalize_img(padded).transpose(2, 0, 1)).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)  # logits (no sigmoid in the model)
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    return prob[:h, :w]


def labels_to_boundaries(labels: np.ndarray) -> np.ndarray:
    """GT boundary mask from a labeled image (same convention as training data)."""
    b = np.zeros(labels.shape, dtype=np.uint8)
    if labels.max() == 0:
        return b
    for shift in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        shifted = np.roll(labels, shift, axis=(0, 1))
        b |= ((labels != shifted) & (labels > 0) & (shifted > 0)).astype(np.uint8)
    for shift in [(1, 0), (0, 1)]:
        shifted = np.roll(labels, shift, axis=(0, 1))
        b |= ((labels > 0) & (shifted == 0)).astype(np.uint8)
        b |= ((labels == 0) & (shifted > 0)).astype(np.uint8)
    return (b > 0).astype(np.uint8)


def boundary_metrics(pred_bin, gt_bin, valid):
    """Dice / IoU of binary boundary maps, computed only inside the layer."""
    p = pred_bin[valid].astype(bool)
    g = gt_bin[valid].astype(bool)
    inter = (p & g).sum()
    dice = 2 * inter / (p.sum() + g.sum()) if (p.sum() + g.sum()) else 1.0
    union = (p | g).sum()
    iou = inter / union if union else 1.0
    return dice, iou


for sample_id in split['val'][:3]:
    img = tifffile.imread(DATA_DIR / 'images' / f'{sample_id}.tif')
    if img.ndim == 3:
        img = img[..., 0]
    gt_labels = tifffile.imread(DATA_DIR / 'labels' / f'{sample_id}.tif')

    # Layer validity: outside the layer the prediction is undefined.
    lm_path = DATA_DIR / 'layer_masks' / f'{sample_id}.tif'
    if lm_path.exists():
        valid = tifffile.imread(lm_path) > 127
    else:
        print(f'  (no layer mask for {sample_id} — evaluating the whole image)')
        valid = np.ones(img.shape[:2], dtype=bool)

    pred_boundary = predict_boundaries(img)
    pred_boundary[~valid] = 0
    pred_binary = (pred_boundary > 0.5).astype(np.uint8)

    gt_boundary = labels_to_boundaries(gt_labels)
    dice, iou = boundary_metrics(pred_binary, gt_boundary, valid)

    # Boundaries -> grains: dilate to connect, interiors are connected components.
    boundary_dilated = cv2.dilate(pred_binary, np.ones((3, 3), np.uint8), iterations=1)
    grain_interior = (boundary_dilated == 0) & valid
    pred_labels, n_grains = ndi.label(grain_interior)

    gt_n = len(np.unique(gt_labels)) - 1
    print(f'{sample_id}: predicted {n_grains} grains vs {gt_n} in GT | '
          f'boundary Dice={dice:.3f}, IoU={iou:.3f}')

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img, cmap='gray'); axes[0].set_title('Input')
    axes[1].imshow(pred_boundary, cmap='hot'); axes[1].set_title('Predicted boundaries')
    axes[2].imshow(pred_labels, cmap='nipy_spectral'); axes[2].set_title(f'Predicted grains ({n_grains})')
    axes[3].imshow(gt_labels, cmap='nipy_spectral'); axes[3].set_title(f'Ground truth ({gt_n})')
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


## 7. Download the trained model

Download `unet_grain_boundaries.pth` and place it in `~/.cache/tem_grain_analyzer/`. The Streamlit app can then load it as a new segmentation method.

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)